In [1]:
chr(0)

'\x00'

(a) It returns \x00

In [2]:
repr(chr(0))

"'\\x00'"

(b) The string representation is '\\x00', which has an additional \ and is enclosed with single quotes

In [3]:
chr(0)

'\x00'

In [4]:
print(chr(0))

 


In [5]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [6]:
print("this is a test" + chr(0) + "string")

this is a test string


(c) when using ()print, the character does not get printed to the output - we have empty print. Without print(), the character is represented as \x00

In [7]:
test_string = "hello! こんにちは！"
utf8_encoded = test_string.encode('utf-8')
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-8'))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf\xef\xbc\x81'
<class 'bytes'>
13
25
hello! こんにちは！


In [8]:
test_string = "hello! こんにちは！"
utf8_encoded = test_string.encode('utf-16')
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-16'))

b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00S0\x930k0a0o0\x01\xff'
<class 'bytes'>
13
28
hello! こんにちは！


In [9]:
test_string = "hello! こんにちは！"
utf8_encoded = test_string.encode('utf-32')
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-32'))

b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00\x01\xff\x00\x00'
<class 'bytes'>
13
56
hello! こんにちは！


2.(a)
We can see that in utf-16 and utf-32, there are a lot of "padding" bytes \x00, which we saw earlier does not represent actual characters. They occur a lot but do not carry meaningful semantic information. The high frequency of their occurence can cause them to take a major part in the tokenizer dictionary, and making it hard to merge those byte combinations with actual meaningful information, leading to inefficient tokenization.



In [20]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join(bytes([b]).decode('utf-8') for b in bytestring)


decode_utf8_bytes_to_str_wrong("你好".encode('utf-8'))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

2.(b)

The above example produces incorrect output. The function incorrectly assumes each utf8 encoded byte corresponding to a character and tries to directly decode it. When then encoded character corresponds to more than 1 byte, the function produces incorrect output.

In [44]:
b = bytes([0xD8, 0xD8])

In [45]:
b.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte

2.(c)

The byte sequence 0xC0, 0x37 cannot be decoded to any Unicode characters because in UTF-8, the byte must follow certain pattern (e.g. the second byte starts with a continuation byte) to allow a low-bit representation of unicode. With 0xD8 0xD8, it is expected that the second byte starts with a continuation byte and shall give an error when it does not.

(train_bpe_tinystories)

(a)
The training took a total of 2 minutes and a peak memory of 621MB. The longest token in the vocab is " accomplishment" (2 other tokens share the same length as it), and it does make sense.

(b)
The step that took the most time was the pretoknization step, specifically, when matching for the pretokenization regex and encoding them into utf-8 bytes tuples. This specific line of code took 78% of my total runtime.

(train_bpe_expts_owt)
(a) The longest token in the vocabulary is '----------------------------------------------------------------', which is a sequence of 64 '-'

(b) Common short sequence like ' a', ' t' appear very early in the trained tokenizer for both datasets. Yet OpenWebText contains many more tokens that consist of unprintable chars, while the tokenizer of TinyStories almost only consists of normal English characters.

(tokenizer_experiments)

(a)
The compression ratio of TinyStories is around 3.98 bytes / token. The compression ratio of the OpenWebText tokenizer is around 

In [4]:
import sys
sys.path.insert(0, "/Users/paulzhu/learn/assignment1-basics")
from answer.transformers.transformer_lm import TransformerLM


llm = TransformerLM(
    d_model=1600,
    num_heads=25,
    d_ff=6400,
    vocab_size=50257,
    context_length=1024,
    num_layers=48)

print(f"Number of parameters: {sum(p.numel() for p in llm.parameters()):,}")

Number of parameters: 2,127,057,600


(transformer_accounting)

(a)
The number of parameters can be decomposed into the following parts:

- TransformerLM
    - Emebedding layer: vocab_size * d_model = 50257 * 1600 = 80411200
    - Transformer Block:
        - MHA:
            - down projection: 3 * num_heads * d_k * d_model = 3 * 25 * 64 * 1600 = 7680000
            - up projection: d_model * num_heads * d_v = 1600 * 25 * 64 = 2560000
            - total = 10240000
        - SwiGLU:
            - 3 Linear Layers: 3 * d_model * d_ff = 3 * 1600 * 6400 = 30720000
        - 2 * RMSNorm:
            - 2 * d_model = 3200
        - Total (48 layers):
            - (10240000 + 30720000 + 3200) * 48 = 1966233600
    - RMSNorm
        - 1600
    - Linear
        - d_model * vocab_size = 1600 * 50257 = 80411200
    - Total = 80411200 + 1966233600 + 1600 + 80411200 = 2127057600 (2.1B)

Using FP16, the memory required to load the model weights is 2127057600 * 4 bytes = 8508230400 bytes (~8GB)


(b)

The matrix multiplicaiton requires in each individual components are:

- Embedding layer: None
- Transformers block
    - RMSNorm: None
    - Multi-head attention:
        - down projection $xW_{proj}^T$
            - $x\in (B, T, 3, 1, d_{model}), W^T\in (3, d_{model}, d_k * num_heads)$
            - FLOP = 2 * 1 * 1600 * 64 * 25 * 3 * 1024 = 15,728,640,000
        - Scaled product attention:
            - $QK^T$
                - $Q\in (B, h, T, d_k)$
                - $K^T\in (B, h, d_k, T)$
                - FLOP = 2 * 1024 * 64 * 1024 * 25 = 3,355,443,200
            - $\text{weight}\cdot V$
                - weight$\in (B, h, T, T)$
                - $V\in (B, h, T, d_v)$
                - FLOP = 2 * 1024 * 1024 * 64 * 25 = 3,355,443,200
        - up projection $\text{attn}\cdot W_O^T$
            - weight$\in (B, T, h*d_v)$
            - $W_O^T\in (h*d_v, d_{model})$
            - FLOP = 2 * 1024 * 1600 * 1600 = 5,242,880,000
        - FLOP = 15728640000 + 3355443200 + 3355443200 + 5242880000 = 27,682,406,400
    - FFN:
        - $(\text{SiLU}(xW_1^T)\odot (xW_2^T))W_2^T$
            - $x\in (B, T, d)$
            - $W_1^T\in (d_{model}, d_{ff})$
            - FLOP = 2 * 1024 * 1600 * 6400 = 20971520000
            - $x\in (B, T, d)$
            - $W_3^T\in (d_{model}, d_{ff})$
            - FLOP = 2 * 1024 * 1600 * 6400 = 20971520000
            - $x\in (B, T, d)$
            - $W_2^T\in (d_{ff}, d_{model})
            - FLOP = 2 * 1024 * 6400 * 1600  = 20971520000
        - FLOP = 3 * 20971520000 = 62,914,560,000
    - FLOP = 27682406400 + 62914560000 = 90,596,966,400
    - FLOP (48 layers) = 90596966400 * 48 = 4,348,654,387,200
- Linear layer $xW^T$
    - $x\in (B, T, d_{model})$
    - $W^T\in (d, \text{vocab\_size})$
    - FLOP = 2 * 1024 * 1600 * 50257 = 164,682,137,600
- FLOP = 4348654387200 + 164682137600 = 4,513,336,524,800


(c)
Looking at the analysis above, we can see that the FFN in each transformer block contributed 62B flop, multiplying by 48 layers, we find that all FFN layers contributed 2.9T FLOPs in total, accounting for the most FLOPs in this LLM architecture.


(d)

Using the same calculation above we can get the FLOP calculation result for the following models:


GPT-2 small:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 3,623,878,656
    - QK^T: 1,610,612,736
    - Weighted V: 1,610,612,736
    - Up projection: 1,207,959,552
    - Total Attention: 8,053,063,680
  FFN (SwiGLU):
    - xW_1^T: 10,066,329,600
    - xW_3^T: 10,066,329,600
    - result*W_2^T: 10,066,329,600
    - Total FFN: 30,198,988,800
  Total per layer: 38,252,052,480

All Transformer Blocks (12 layers):
  459,024,629,760

Final Linear Layer:
  79,047,426,048

TOTAL FLOPs: 538,072,055,808

GPT-2 medium:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 6,442,450,944
    - QK^T: 2,147,483,648
    - Weighted V: 2,147,483,648
    - Up projection: 2,147,483,648
    - Total Attention: 12,884,901,888
  FFN (SwiGLU):
    - xW_1^T: 13,421,772,800
    - xW_3^T: 13,421,772,800
    - result*W_2^T: 13,421,772,800
    - Total FFN: 40,265,318,400
  Total per layer: 53,150,220,288

All Transformer Blocks (24 layers):
  1,275,605,286,912

Final Linear Layer:
  105,396,568,064

TOTAL FLOPs: 1,381,001,854,976


GPT-2 large:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 10,066,329,600
    - QK^T: 2,684,354,560
    - Weighted V: 2,684,354,560
    - Up projection: 3,355,443,200
    - Total Attention: 18,790,481,920
  FFN (SwiGLU):
    - xW_1^T: 16,777,216,000
    - xW_3^T: 16,777,216,000
    - result*W_2^T: 16,777,216,000
    - Total FFN: 50,331,648,000
  Total per layer: 69,122,129,920

All Transformer Blocks (36 layers):
  2,488,396,677,120

Final Linear Layer:
  131,745,710,080

TOTAL FLOPs: 2,620,142,387,200


GPT-2 XL:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 15,728,640,000
    - QK^T: 3,355,443,200
    - Weighted V: 3,355,443,200
    - Up projection: 5,242,880,000
    - Total Attention: 27,682,406,400
  FFN (SwiGLU):
    - xW_1^T: 20,971,520,000
    - xW_3^T: 20,971,520,000
    - result*W_2^T: 20,971,520,000
    - Total FFN: 62,914,560,000
  Total per layer: 90,596,966,400

All Transformer Blocks (48 layers):
  4,348,654,387,200

Final Linear Layer:
  164,682,137,600

TOTAL FLOPs: 4,513,336,524,800


---
If we look at the Attention part of each model, the number of FLOP from smaller to larger models are:
8,053,063,680 -> 12,884,901,888 -> 18,790,481,920 -> 27,682,406,400

While for FFN, the FLOPs are:
30,198,988,800 -> 40,265,318,400 -> 50,331,648,000 -> 62,914,560,000

For the final linear layer:
79,047,426,048 -> 105,396,568,064 -> 131,745,710,080 -> 164,682,137,600

We can see that eahc model config changes from smaller to larger model brings about 50% increase in attention FLOP, while less than 50% change in FFN FLOPs (1/3 to 1/4 to 1/5). The final linear projection layer observes the same trend of change as the per-layer FFN, so we can conclude that the model config change brings more changes in the attention part for the given set of GPT2 configs.


(e)

GPT-2 XL (long context)

- Per Transformer Layer:
  - Attention:
    - Down projection (QKV): 251,658,240,000
    - QK^T: 858,993,459,200
    - Weighted V: 858,993,459,200
    - Up projection: 83,886,080,000
    - Total Attention: 2,053,531,238,400
  - FFN (SwiGLU):
    - xW_1^T: 335,544,320,000
    - xW_3^T: 335,544,320,000
    - result*W_2^T: 335,544,320,000
    - Total FFN: 1,006,632,960,000
  - Total per layer: 3,060,164,198,400

- All Transformer Blocks (48 layers):
  - 146,887,881,523,200

- Final Linear Layer:
  - 2,634,914,201,600

TOTAL FLOPs: 149,522,795,724,800



We can see that when increasing the context length by 16x, all linear layers (QKV down projection, up projection, FFN, final linear projection) has the same increase of 16x in FLOP. However, the attention calculation including QK^T and weighted V calculation are increased by 256x, causing the FLOP increase in attention to be much larger than linear layers. The total number of FLOP of the model increased by about 33x.

In [1]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = dict(lr=lr)
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p]
                t = state.get("t", 0)
                grad = p.grad.data
                p.data -= lr / math.sqrt(t + 1) * grad
                state["t"] = t + 1

        return loss

In [2]:
weights = torch.nn.Parameter(5 * torch.rand((10, 10)))
opt = SGD([weights], lr=1e1)

for t in range(100):
    opt.zero_grad()
    loss = (weights**2).mean()
    print(loss.cpu().item())
    loss.backward()
    opt.step()



9.012686729431152
5.768118858337402
4.252011299133301
3.32674503326416
2.6946630477905273
2.234184503555298
1.8842384815216064
1.6101353168487549
1.390478491783142
1.21126127243042
1.0628925561904907
0.9385679364204407
0.8333200812339783
0.7434356212615967
0.6660830974578857
0.5990664958953857
0.5406575202941895
0.48947814106941223
0.44441741704940796
0.4045705497264862
0.36919379234313965
0.3376711308956146
0.3094884157180786
0.28421351313591003
0.26148128509521484
0.24098114669322968
0.22244778275489807
0.20565328001976013
0.19040115177631378
0.17652112245559692
0.16386520862579346
0.15230423212051392
0.1417250633239746
0.13202840089797974
0.12312664091587067
0.11494248360395432
0.10740736126899719
0.10046041756868362
0.0940474346280098
0.08812002837657928
0.08263495564460754
0.0775534063577652
0.07284056395292282
0.06846509873867035
0.06439873576164246
0.06061597913503647
0.05709375441074371
0.05381115525960922
0.05074920505285263
0.04789067804813385
0.04521988704800606
0.0427225306

In [3]:
weights = torch.nn.Parameter(5 * torch.rand((10, 10)))
opt = SGD([weights], lr=1e2)

for t in range(100):
    opt.zero_grad()
    loss = (weights**2).mean()
    print(loss.cpu().item())
    loss.backward()
    opt.step()



7.940154552459717
7.9401535987854
1.3623149394989014
0.03260326385498047
4.996017044226352e-17
5.5683715375901955e-19
1.8750664219084144e-20
1.116990085237265e-21
9.582261861283422e-23
1.0646958237206728e-23
1.4382865039244574e-24
2.266609880106311e-25
4.048909331030112e-26
8.028658825906069e-27
1.739564251830694e-27
4.068337784901e-28
1.01708444622525e-28
2.696815603769485e-29
7.535258083460284e-30
2.2068010599399442e-30
6.743385561712466e-31
2.1417304886953737e-31
7.046620786427986e-32
2.394831996067957e-32
8.385984045605892e-33
3.0189549176336934e-33
1.1151462157254983e-33
4.219131619251876e-34
1.6325009552774389e-34
6.450823715499816e-35
2.5999182845163515e-35
1.0675552851735484e-35
4.461241770815589e-36
1.8955883143665232e-36
8.182358857318498e-37
3.585201045446141e-37
1.5934227616008697e-37
7.178563925771773e-38
3.276134355582605e-38
1.5137431372573262e-38
7.077421845992815e-39
3.346672076900709e-39
1.599790990497963e-39
7.727460381519204e-40
3.770123453342704e-40
1.8571828937236

In [4]:
weights = torch.nn.Parameter(5 * torch.rand((10, 10)))
opt = SGD([weights], lr=1e3)

for t in range(100):
    opt.zero_grad()
    loss = (weights**2).mean()
    print(loss.cpu().item())
    loss.backward()
    opt.step()



7.221335411071777
2606.902099609375
450253.0
50085840.0
4056953088.0
256040189952.0
13144267161600.0
565522571722752.0
2.084395732382515e+16
6.693226181673615e+17
1.8975888442623263e+19
4.801503594784311e+20
1.0940864790748788e+22
2.2620483851412774e+23
4.270968026087321e+24
7.405307057810147e+25
1.1848491292496235e+27
1.7568925799449707e+28
2.4234803222875203e+29
3.1204741876545475e+30
3.7619585768525397e+31
4.258124557741858e+32
4.536515763368957e+33
4.559528264038247e+34
4.33232784117056e+35
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf
inf


loss decays faster when using a lr of 1e2 than 1e1, while optimizer with lr=1e3 gives diverging loss.

Problem (adamwAccounting)

(a)

To account for the memory usage that AdamW requires for back-propagation, we need to consider the weight (learnable parameters) storage, the activation storage, gradient storage and optimizer states of the model.

The variables available for use in our algebraic expressions are:
batch_size and the model hyperparameters (vocab_size, context_length, num_layers, d_model, num_heads). Assume d_ff = 4 × d_model.

For the weight part, we would conisder the entire model:

- Token Embedding
    - weights: vocab_size * d_model
- Transformer Block * num_layers
    - RMSNorm * 2
        - weights: d_model
        - total: d_model * 2
    - MHA:
        - QKV proj weights: 3 * d_model * d_model
        - Up proj weights: d_model * d_model
        - total: 4 * d_model^2
    - FFN:
        - Linear layer * 3:
            - weights: d_model * 4 * d_model (consider d_ff as 4 * d_model)
            - total: 12 * d_model^2
    - total: (16 * d_model^2 + 2 * d_model) * num_layers
- RMS Norm:
    - weights: d_model
- Linear layer:
    - weights: d_model * vocab_size
- Total: ((2 + 16 * d_model) * num_layers + 2 * vocab_size + 1) * d_model


For the activation parts, we need to store both the input and output activations of a computation for gradient calculation. For simplicity, we only consider the following activations:

- Transformer block * num_layers
    - RMSNorm * 2
        - input x: batch_size * context_length * d_model
        - output: batch_size * context_length * d_model
        - total: 4 * batch_size * context_length * d_model
    - MHA
        - QKV proj
            - input x: batch_size * context_length * d_model
            - output: 3 * batch_size * cotext_length * d_model
            - total: 4 * batch_size * context_length * d_model
        - Q^TK
            - input Q: batch_size * context_length * d_model
            - input K: batch_size * context_length * d_model
            - output: batch_size * num_heads * context_length * context_length
            - total: batch_size * context_length * (2 * d_model + num_heads * context_length)
        - softmax
            - input x: batch_size * num_heads * context_length * context_length
            - output: batch_size * num_heads * context_length * context_length
            - total: 2 * batch_size * num_heads * context_length * context_length
        - weighted sum of V:
            - input weights: batch_size * num_heads * context_length * context_length
            - input V: batch_size * context_length * d_model
            - output: batch_size * context_length * d_model
            - total: batch_size * context_length * (2 * d_model + num_heads * context_length)
        - output projection
            - input weighted_sum: batch_size * context_length * d_model
            - output: batch_size * context_length * d_model
            - total: 2 * batch_size * context_length * d_model
        - total: 10 * batch_size * context_length * d_model + 4 * batch_size * num_heads * context_length^2
    - FFN
        - W1 matrix multiply
            - input x: batch_size * context_length * d_model
            - output: batch_size * context_length * 4 * d_model
            - total: 5 * batch_size * context_length * d_model
        - SiLU:
            - input x: batch_size * context_length * 4 * d_model
            - output: batch_size * context_length * 4 * d_model
            - total: 8 * batch_size * context_length * d_model
        - W2 matrix multiply
            - input x: batch_size * context_length * 4 * d_model
            - output: batch_size * context_length * d_model
            - total: 5 * batch_size * context_length * d_model
        - total: 18 * batch_size * context_length * d_model
    - total: (32 * batch_size * context_length * d_model + 4 * batch_size * num_heads * context_length^2) * num_layers

- final RMS Norm
    - input x: batch_size * context_length * d_model
    - output: batch_size * context_length * d_model
    - total: 2 * batch_size * context_length * d_model
- output embedding
    - input x: batch_size * context_length * d_model
    - output: batch_size * context_length * vocab_size
    - total: batch_size * context_length * d_model + batch_size * context_length * vocab_size
- cross entropy on logits
    - input x: batch_size * context_length * vocab_size
    - output: scaler
    - total: batch_size * context_length * vocab_size
- total: (32 * num_layers + 3) * batch_size * context_length * d_model + 4 * num_layers * batch_size * num_heads * context_length^2 + 2 * batch_size * context_length * vocab_size



For gradients, since the shape is exactly the same as model parameters, so we will also have:
```
((2 + 16 * d_model) * num_layers + 2 * vocab_size + 1) * d_model
```
for gradients memory

For optimizer states, for each parameter tensor, we have t, m, v as states for that tensor, where t is scaler and m and v are tensors of the same shape as that paramter tensor. Considering that t is total overshadowed by the other two states in terms of memory consumption, I will ignore it in my calculation, and since m and v has the same shape as the param tensor, we can again reuse our previous calculation for model weights and conclude that the memory consumption for optimizer states are:
```
2 * ((2 + 16 * d_model) * num_layers + 2 * vocab_size + 1) * d_model
```


Final total memory consumption when using float32 (4 bytes per scaler) for every tensor:

4 * (64 * num_layers * d_model^2 + (8 * num_layers + 8 * vocab_size + 4) * d_model + (32 * num_layers + 3) * batch_size * context_length * d_model + 4 * num_layers * batch_size * num_heads * context_length^2 + 2 * batch_size * context_length * vocab_size)


(b)

When using the GPT-2 XL model configuration, we have the following hyperparams:
vocab_size: 50257
context_length: 1024
num_layers: 48
d_model: 1600
num_heads: 25
d_ff: 6400

Plugging in these values, the peak memory consumption for GPT-2 XL is
4 * (64 * 48 * 1600^2 + (8 * 48 + 8 * 50257 + 4) * 1600 + (32 * 48 + 3) * batch_size * 1024 * 1600 + 4 * 48 * batch_size * 25 * 1024^2 + 2 * batch_size * 1024 * 50257)
= 34032921600 + 30630354944 * batch_size

Assuming the total available memory is 80GB (80 * 1024 * 1024 * 1024 = 85899345920)

The max batch size we can afford is:
(85899345920 - 34032921600) / 30630354944 = 1.69

We can only afford a batch size of 1

(c)

Operations inside the step() method of AdamW optimizers does not include any matrix multiplication, so we only consider the element wise operations here. All of m, v, grad are of the same shape as the parameter tensor, so we will enumerate all element-wise operations on those and multiply the number of weights by that count to get the total FLOPs.
1. beta1 * m
2. (1 - beta1) * grad
3. beta1 * m + (1 - beta1) * grad
4. beta2 * v
5. grad ** 2
6. (1- beta2) * grad ** 2
7. beta2 * v + (1- beta2) * grad ** 2
8. a_t * m
9. torch.sqrt(v)
10. torch.sqrt(v) + eps
11. a_t * m / (torch.sqrt(v) + eps)
12. p.data -= a_t * m / (torch.sqrt(v) + eps)
13. lr * weight_decay * p.data
14. p.data -= lr * weight_decay * p.data

Hence, the total FLOP will be given by 14 * number of params in the model
Which is 14 * ((2 + 16 * d_model) * num_layers + 2 * vocab_size + 1) * d_model


(d)

The number of FLOPs in the model forward pass of GPT-2 XL, as computed from 2(e), need to take 4513336524800 FLOPs in its forward pass for a batch size of 1. When forward passing with a batch size of 1024, the FLOP needed is 4621656601395200. Assuming back propagation requires twice the FLOP as forward pass, the FLOP needed for backward pass is 9243313202790400, and the total FLOP including both forward and backward pass is 13864969804185600. The number of FLOP needed in the AdamW optimizer step is 14 * ((2 + 16 * 1600) * 48 + 2 * 50257 + 1) * 1600 = 29778806400

The number of FLOP needed for a single step (including forward / backward) pass with batch size of 1024 is 13864969804185600 + 29778806400 = 13864999582992000. To run 400k steps, the total number of FLOPs are 13864999582992000 * 400000 = 5.54599983e21

For a A100 GPU with 19.5 teraFLOP / s for float32 operations and with a MFU of 50%, it can process 9.75e12 FLOPs / s.

The number of seconds needed for training GPT-2 XL for 400k steps with 1024 batch size on a A100 device is 5.54599983e21 / 9.75e12 = 568820495.385, which is approximately 6583.57 days
